# OP2 Reader — Test Notebook

In [ ]:
# Setup — reload module and open file
import sys
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

sys.path.insert(0, r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader")
from op2_native import OP2

OP2_FILE = r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader\testing-015.op2"
op2 = OP2(OP2_FILE)
print("Loaded:", OP2_FILE)


In [ ]:
## 1 — File Summary
# All records scanned from the binary file.

In [ ]:
summary = op2.summary()
print(f"{len(summary)} records total")
summary


In [ ]:
## 2 — Displacements (OUGV1)
# Columns: GRID, CP (output coord system), DX, DY, DZ (translations), RX, RY, RZ (rotations)

In [ ]:
displ = op2.displacements()
for sc, df in displ.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df.head(10))


In [ ]:
## 3 — Element Stresses (OES1X1 shell)
# Centroid-only:  EID, FD1, SX1, SY1, TXY1, ANG1, MAJOR1, MINOR1, VM1,
#                     FD2, SX2, SY2, TXY2, ANG2, MAJOR2, MINOR2, VM2
# With corners:   EID, GRID (0=centroid), FD1..VM2  →  5 rows/element
# VM1/VM2 = Von Mises (or Max Shear, depending on Nastran STRESS case control)

In [ ]:
stresses = op2.stresses()
for sc, df in stresses.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df)


In [ ]:
## 4 — Element Forces (OEF1)
# CQUAD4/CTRIA3: EID, NX, NY, NXY (membrane), MX, MY, MXY (bending), QX, QY (transverse shear)
# CBAR/CBEAM:   EID, BM1A, BM2A, BM1B, BM2B, TS1, TS2, AF, TRQ

In [ ]:
eforces = op2.element_forces()
for sc, df in eforces.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df.head(10))


In [ ]:
## 5 — SPC Constraint Forces (OQG1)

In [ ]:
spc = op2.spc_forces()
for sc, df in spc.items():
    nonzero = df[(df[["FX","FY","FZ","MX","MY","MZ"]].abs() > 0).any(axis=1)]
    print(f"Subcase {sc}: {len(df)} rows total, {len(nonzero)} nonzero")
    display(nonzero.head(20))


In [ ]:
## 6 — Applied Loads (OPG1)

In [ ]:
try:
    loads = op2.applied_loads()
    if loads:
        for sc, df in loads.items():
            print(f"Subcase {sc}: {len(df)} rows")
            display(df.head(10))
    else:
        print("No OPG1 tables found in this file.")
except Exception as e:
    print(f"Error: {e}")


In [ ]:
## 7 — Strains (OSTR1)

In [ ]:
try:
    strains = op2.strains()
    if strains:
        for sc, df in strains.items():
            print(f"Subcase {sc}: {len(df)} rows")
            display(df.head(10))
    else:
        print("No OSTR1 tables found in this file.")
except Exception as e:
    print(f"Error: {e}")


In [ ]:
## 8 — Solid / Bar Stresses
# Returns empty for this file (no solid/bar elements), but confirms the API works.

In [ ]:
solid = op2.solid_stresses()
bars  = op2.bar_stresses()
print(f"solid_stresses subcases : {sorted(solid.keys())}")
print(f"bar_stresses   subcases : {sorted(bars.keys())}")
if not solid and not bars:
    print("(none — no solid/bar elements in this file)")


In [ ]:
## 8b — Eigenvalues (LAMA)
# Available for modal (SOL 103) and buckling (SOL 105) analyses.
# Columns: MODE, ORDER, EIGENVALUE (ω², rad²/s²), RADIANS (ω, rad/s),
#          CYCLES (f, Hz), GENM (generalised mass), GENSTIF (generalised stiffness)
eigs = op2.eigenvalues()
if eigs:
    for sc, df in eigs.items():
        print(f"Subcase {sc}: {len(df)} modes")
        display(df)
else:
    print("No LAMA table found (this is a static analysis file).")

In [ ]:
## 9 — Summary Table
import pandas as pd

all_results = {
    "displacements":      op2.displacements(),
    "stresses":           op2.stresses(),
    "stresses_corners":   op2.stresses_with_corners(),
    "solid_stresses":     op2.solid_stresses(),
    "bar_stresses":       op2.bar_stresses(),
    "element_forces":     op2.element_forces(),
    "spc_forces":         op2.spc_forces(),
    "applied_loads":      op2.applied_loads(),
    "strains":            op2.strains(),
    "eigenvalues":        op2.eigenvalues(),
}

rows = []
for name, d in all_results.items():
    if d:
        for sc, df in d.items():
            rows.append({"result": name, "subcase": sc, "rows": len(df), "cols": len(df.columns)})
    else:
        rows.append({"result": name, "subcase": "-", "rows": 0, "cols": 0})

# Grid Point Weight Generator (OGPWG) — single result, not subcase-keyed
gw = op2.grid_weight()
if gw:
    rows.append({"result": "grid_weight", "subcase": "-", "rows": 1, "cols": len(gw["summary"].columns)})
else:
    rows.append({"result": "grid_weight", "subcase": "-", "rows": 0, "cols": 0})

pd.DataFrame(rows)

In [ ]:
## 10 — Grid Point Weight (OGPWG)
# mass, centre of gravity, and inertia matrix from the OGPWG table.

In [ ]:
gw = op2.grid_weight()
if gw:
    print(f"Total mass  : {gw['mass']:.6g}")
    print(f"CG (X, Y, Z): {[round(v, 6) for v in gw['cg']]}")
    display(gw["summary"])
else:
    print("No OGPWG table found in this file.")

In [ ]:
## 11 — Corner Stresses (OES1X1 with corner nodes)
# 5 rows per CQUAD4: GRID=0 is centroid, GRID!=0 are the 4 corner node IDs.
sc_corners = op2.stresses_with_corners()
for sc, df in sc_corners.items():
    centroids = df[df["GRID"] == 0]
    corners   = df[df["GRID"] != 0]
    print(f"Subcase {sc}: {len(df)} rows  ({len(centroids)} centroids + {len(corners)} corner rows)")
    display(df.head(10))

In [ ]:
## 12 — find_by_eid  &  to_csv

# -- 12a: look up all results for a single element --
EID_QUERY = 1
hits = op2.find_by_eid(EID_QUERY)
print(f"Results found for EID={EID_QUERY}:")
for result_name, df in hits.items():
    print(f"  {result_name}: {len(df)} row(s)")
    display(df)

# -- 12b: export every result table to CSV --
import tempfile, os
out_dir = r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader\csv_export"
written = op2.to_csv(out_dir)
print(f"\nCSV files written to: {out_dir}")
for fname, fpath in sorted(written.items()):
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<35s}  {size_kb:7.1f} kB")

In [ ]:
# §13 — Performance: cache speedup
import time, importlib, op2_native.reader as _r
importlib.reload(_r)
from op2_native.reader import OP2

fresh = OP2(OP2_FILE)

# First call — cold (decode from bytes)
t0 = time.perf_counter()
_ = fresh.stresses()
cold_ms = (time.perf_counter() - t0) * 1000

# Second call — warm (from cache)
t0 = time.perf_counter()
_ = fresh.stresses()
warm_ms = (time.perf_counter() - t0) * 1000

print(f"stresses()  cold: {cold_ms:.2f} ms")
print(f"stresses()  warm: {warm_ms:.2f} ms  ({cold_ms/max(warm_ms,0.001):.0f}× speedup)")

# Confirm cache is populated
print(f"\nCache keys: {list(fresh._cache.keys())}")
fresh.clear_cache()
print(f"After clear_cache(): {list(fresh._cache.keys())}")

In [ ]:
# §14 — find_by_grid(grid_id): nodal results lookup
GRID_QUERY = 51   # pick any node ID from the model

grid_hits = op2.find_by_grid(GRID_QUERY)
print(f"find_by_grid({GRID_QUERY}) found results in: {list(grid_hits.keys())}\n")
for name, df in grid_hits.items():
    print(f"--- {name} ({len(df)} rows) ---")
    display(df.head(6))

In [ ]:
## 15 — OP2 repr  &  Results object

# The OP2 repr now shows a summary of all non-empty result types
print(repr(op2))
print()

# Results gives flat attribute access to all tables for one subcase
from op2_native import Results
r = op2.results(subcase=1)
print(repr(r))
print()

# Access individual tables directly as attributes
print("Stresses (first 3 rows):")
display(r.stresses.head(3))

print("Element forces (first 3 rows):")
display(r.element_forces.head(3))

In [ ]:
## 16 — Plotly Visualisations
from op2_native import plots

r = op2.results(subcase=1)

# Von Mises stress — max of both fibers, coloured bar chart sorted by EID
plots.plot_vm_stress(r.stresses, fiber="max")

In [ ]:
# Displacement magnitude scatter — each node coloured by |U|
plots.plot_displacement_magnitude(r.displacements, component="mag")

In [ ]:
# Element force NX (membrane x) — diverging red-blue colorscale
plots.plot_element_forces(r.element_forces, component="NX")

In [ ]:
# VM1 stress histogram with mean line
plots.plot_stress_histogram(r.stresses, column="VM1", bins=40)

## §17 — envelope(), describe(), and new plots

In [ ]:
# Reload modules so changes to reader.py and plots.py are picked up
import importlib, op2_native, op2_native.reader, op2_native.plots
importlib.reload(op2_native.plots)
importlib.reload(op2_native.reader)
importlib.reload(op2_native)

from op2_native import OP2
op2 = OP2(OP2_FILE)
r   = op2.results(1)
print("Modules reloaded OK")

In [ ]:
# envelope() — worst-case VM1 stress across all subcases
env = op2.envelope("stresses", "VM1", "absmax")
print(f"Envelope shape: {env.shape}")
print(f"Governing column range: {env['VM1'].min():.4g} → {env['VM1'].max():.4g}")
display(env.head(10))

In [ ]:
# describe() — statistical summary of all non-empty result tables
desc = op2.describe()
print(f"describe() shape: {desc.shape}")
display(desc.round(4))

In [ ]:
# Top-20 stressed elements — horizontal bar chart with EID labels
op2_native.plots.plot_top_n_stress(r.stresses, n=20, fiber="max")

In [ ]:
# Principal stress comparison: MAJOR vs MINOR vs VM (fiber 1, top 50 elements by VM)
op2_native.plots.plot_principal_stress(r.stresses, fiber="1", max_elements=50)

In [ ]:
# Polar scatter of in-plane element forces: NX at 0°, NY at 90°, NXY at 45°
op2_native.plots.plot_forces_polar(r.element_forces)

## §18 — subcases(), eqexin(), plot_stress_summary()

In [ ]:
import importlib, op2_native, op2_native.reader, op2_native.plots
importlib.reload(op2_native.plots)
importlib.reload(op2_native.reader)
importlib.reload(op2_native)
from op2_native import OP2
op2 = OP2(OP2_FILE)
r   = op2.results(1)
print("Reloaded")

In [ ]:
# subcases() — cross-tab of result type vs subcase ID
sc_table = op2.subcases()
print(sc_table.to_string())
display(sc_table)

In [ ]:
# eqexin() — GRID to DOF-pointer mapping
eq = op2.eqexin()
print(f"EQEXIN: {len(eq)} rows, GRID range {eq['GRID'].min()}–{eq['GRID'].max()}")
print(f"Constrained grids (EQTYPE==0): {(eq['EQTYPE']==0).sum()}")
display(eq.head(10))

In [ ]:
# Stress component summary — 4×2 histogram grid for all components
op2_native.plots.plot_stress_summary(r.stresses, fiber="1")

## §19 — Multi-record data fix verification

**Problem (fixed 2026-03-30):** All three data decoders (`decode_oes1x1_shell`, `decode_oes1x1_shell_corners`, `decode_oef1`) only read the **first** Fortran data record per table, silently dropping the remaining records. For a ~3,661-element model the OES1X1 table has 5 data records and OEF1X has 3, so ~80% of elements were missing from all outputs.

**Fix:** `collect_data_records_after()` + `load_data_bytes()` added to `oes_peek.py`. Both shell decoders and the force decoder now concatenate all contiguous data records before decoding.

In [ ]:
# §19a — Verify full element counts after multi-record fix
import sys
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

from op2_native import OP2
op2 = OP2(OP2_FILE)
r = op2.results(1)

st = r.stresses
sc = r.stresses_corners
ef = r.element_forces

print(f"stresses()        rows: {len(st):>6,}   unique EIDs: {st['EID'].nunique():>5,}")
print(f"stresses_corners  rows: {len(sc):>6,}   unique EIDs: {sc['EID'].nunique():>5,}")
print(f"element_forces()  rows: {len(ef):>6,}   unique EIDs: {ef['EID'].nunique():>5,}")
print()
print(f"stresses   attrs: {st.attrs.get('all_data_records')}")
print(f"forces     attrs: {ef.attrs.get('all_data_records')}")


## §19b — Next steps

- **`oes_bar.py`** — `decode_oes_bar` still uses `first_stress_record_after` + single `rec.data`. Update to `load_data_bytes` (same pattern, but bar models are often small so may not be urgent).
- **`lama.py`** — `_first_data_record_after` is its own local helper; verify that eigenvalue tables never span multiple records (they typically don't for modal analyses).
- **Solid elements** — no solid-element OES decoder exists yet; if needed, add `oes1x1_solid.py` following the same `load_data_bytes` pattern.
- **OUG / grid-point results** — already single-record in typical models; no change needed unless very large grids are encountered.
- **Test with pyNastran** — cross-check EID sets and stress values against `pyNastran.op2.OP2` as a reference implementation.

In [ ]:
EIDS_TO_CHECK = [417, 418, 419] # CBEAM element IDs
GRIDS_TO_CHECK = [1, 2, 3] # node IDs

import sys, warnings

from matplotlib.pyplot import title
warnings.filterwarnings("ignore")
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

from op2_native import OP2
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.6e}".format)

CYL_OP2 = r"run_files/model/cylinder-0000.op2"
cyl = OP2(CYL_OP2)
print("Loaded:", CYL_OP2)

all_sc = sorted(
set(cyl.displacements()) | set(cyl.bar_stresses()) |
set(cyl.element_forces()) | set(cyl.bush_stresses()) |
set(cyl.bush_forces()) | set(cyl.solid_stresses())
)
print(f"Subcases found: {all_sc}\n")



def section(title):
    print(f"\n{'═'*72}\n {title}\n{'═'*72}")

for sc in all_sc:
    print(f"\n\n{'#'*72}")
    print(f" SUBCASE {sc}")
    print(f"{'#'*72}")

    # ── Displacements ─────────────────────────────────────────────────────────
    disp = cyl.displacements().get(sc)
    if disp is not None:
        d = disp[disp["GRID"].isin(GRIDS_TO_CHECK)]
        if len(d):
            section("DISPLACEMENTS  (T1/T2/T3 = translations,  R1/R2/R3 = rotations)")
            display(d.set_index("GRID").drop(columns=["CP"], errors="ignore"))

    # ── CBEAM stresses ────────────────────────────────────────────────────────
    bstr = cyl.bar_stresses().get(sc)
    if bstr is not None:
        b = bstr[bstr["EID"].isin(EIDS_TO_CHECK)]
        if len(b):
            section("CBEAM STRESSES  (SXC/SXD/SXE/SXF at 4 fiber corners,  SMAX/SMIN)")
            display(b.set_index(["EID", "GRID"]))

    # ── CBEAM forces ──────────────────────────────────────────────────────────
    ef = cyl.element_forces().get(sc)
    if ef is not None:
        e = ef[ef["EID"].isin(EIDS_TO_CHECK)]
        if len(e):
            section("CBEAM FORCES  (BM1/BM2 = bending moments,  WS1/WS2 = web shears,  AF = axial,  TRQ = torque)")
            display(e.set_index(["EID", "GRID"]))

    # ── CBUSH stresses ────────────────────────────────────────────────────────
    bush_s = cyl.bush_stresses().get(sc)
    if bush_s is not None:
        bs = bush_s[bush_s["EID"].isin(EIDS_TO_CHECK)]
        if len(bs):
            section("CBUSH STRESSES  (EX/EY/EZ = axial/shear,  ETX/ETY/ETZ = rotational)")
            display(bs.set_index("EID"))

    # ── CBUSH forces ──────────────────────────────────────────────────────────
    bush_f = cyl.bush_forces().get(sc)
    if bush_f is not None:
        bf = bush_f[bush_f["EID"].isin(EIDS_TO_CHECK)]
        if len(bf):
            section("CBUSH FORCES  (FX/FY/FZ = forces,  MX/MY/MZ = moments)")
            display(bf.set_index("EID"))

    # ── CTETRA stresses ───────────────────────────────────────────────────────
    sol = cyl.solid_stresses().get(sc)
    if sol is not None:
        ss = sol[sol["EID"].isin(EIDS_TO_CHECK)]
        if len(ss):
            section("CTETRA STRESSES  (GRID=0 → centroid,  GRID!=0 → corner node)")
            display(ss.set_index(["EID", "GRID"]))

    print("\n\nDone. Adjust EIDS_TO_CHECK / GRIDS_TO_CHECK at the top and re-run the two cells.")

In [ ]:
## §20 — Nonlinear Results (`cylinder-0001.op2`)

# This section exercises all result types on the nonlinear (SOL 106) run file.
# The model contains `CBEAM` + `CTETRA` elements (no `CBUSH`), so `bush_stresses` / `bush_strains` return empty dicts for this file.
# Linear-equivalent strains (`bar_strains`, `solid_strains`) are also decoded from the `OSTR1X` table.


In [ ]:
# §20a — Load the NL file and show available results
import importlib, op2_native
importlib.reload(op2_native.reader); from op2_native.reader import OP2

NL_OP2 = 'run_files/model/cylinder-0001.op2'
nl = OP2(NL_OP2)

print("All tables (includes metadata blocks):")
print(nl.table_names(results_only=False))
print()
print("Result tables only:")
print(nl.table_names(results_only=True))


In [ ]:
# §20b — Nonlinear bar stresses (OESNLXR CBEAM)
nl_bstr = nl.nl_bar_stresses()
for sc, df in nl_bstr.items():
    print(f"nl_bar_stresses sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
nl_bstr[1].head(4)


In [ ]:
# §20c — Nonlinear solid stresses (OESNLXR CTETRA)
nl_sstr = nl.nl_solid_stresses()
for sc, df in nl_sstr.items():
    print(f"nl_solid_stresses sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
# Show centroid rows only (GRID == 0)
nl_sstr[1][nl_sstr[1]['GRID'] == 0].head(4)


In [ ]:
# §20d — Linear-equivalent bar strains (OSTR1X CBEAM) — NL file
nl_bs = nl.bar_strains()
for sc, df in nl_bs.items():
    print(f"bar_strains (NL) sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
nl_bs[1].head(4)


In [ ]:
# §20e — Linear-equivalent solid strains (OSTR1X CTETRA) — NL file
nl_ss = nl.solid_strains()
for sc, df in nl_ss.items():
    print(f"solid_strains (NL) sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")
nl_ss[1].head(4)


In [ ]:
# §20f — Bar / solid / bush strains on LINEAR file
lin = OP2('run_files/model/cylinder-0000.op2')
lin_bs = lin.bar_strains()
lin_ss = lin.solid_strains()
lin_bu = lin.bush_strains()
for name, d in [('bar_strains', lin_bs), ('solid_strains', lin_ss), ('bush_strains', lin_bu)]:
    for sc, df in d.items():
        print(f"{name} sc={sc}: {len(df)} rows, {df['EID'].nunique()} elements")


In [ ]:
# §20g — Results helper: check all NL subcase 1 result types
r_nl = nl.results(1)
r_nl


In [ ]:
## §21 — OSTR1EL: Element-CS Strains & Bug Fix

# `OSTR1EL` is the element-coordinate-system strain output table, distinct from `OSTR1X` (basic/material CS).
# Previously `bar_strains()` inadvertently included both because `"OSTR1"` substring-matched `"OSTR1EL"`, doubling the row count.
# The fix claims `OSTR1EL` first so it is decoded separately.

In [ ]:
# Bug-fix verification: bar_strains() now returns exactly 832 rows (no duplicates)
bs = lin.bar_strains()
for sc, df in bs.items():
    dups = df.duplicated(subset=["EID", "GRID", "SD"]).sum()
    print(f"bar_strains  sc={sc}: {len(df):>6} rows, {dups} duplicates")

In [ ]:
# New element-CS strain methods on the linear model
bs_el = lin.bar_strains_el()
ss_el = lin.solid_strains_el()
bu_el = lin.bush_strains_el()
for sc in bs_el:
    print(f"bar_strains_el   sc={sc}: {len(bs_el[sc]):>6} rows")
for sc in ss_el:
    print(f"solid_strains_el sc={sc}: {len(ss_el[sc]):>6} rows")
for sc in bu_el:
    print(f"bush_strains_el  sc={sc}: {len(bu_el[sc]):>6} rows")

In [ ]:
import pandas as pd

# Compare basic-CS vs element-CS bar strains for the same element
eid = bs[1]["EID"].iloc[0]
basic   = bs[1][bs[1]["EID"] == eid].assign(coord="basic")
element = bs_el[1][bs_el[1]["EID"] == eid].assign(coord="element")
pd.concat([basic, element]).set_index(["EID", "GRID", "SD", "coord"])

In [ ]:
## §22 — Normal Modes (`cylinder-0002.op2`)

# SOL 103 run with 10 extracted modes.  Available tables: `LAMA` (eigenvalues),
# `OUGV1` (eigenvectors), `OQG1` (SPC forces per mode), `OGPWG` (grid weight).

# **Bug fixed this session**: `decode_lama` was picking up the IDENT header record as
# the data payload, producing garbage (MODE=101, all zeros).  The fix adds content-
# based validation — a record only qualifies as eigenvalue data if its first word is a
# valid mode number (1–100000) and `sqrt(EIGENVALUE) ≈ RADIANS` within 5 %.

In [ ]:
import importlib, op2_native, op2_native.decoders.lama as _lama_mod
importlib.reload(_lama_mod)
importlib.reload(op2_native.reader)
from op2_native.reader import OP2

NM_OP2 = 'run_files/model/cylinder-0002.op2'
nm = OP2(NM_OP2)

print("Tables in file:", nm.table_names(results_only=True))


In [ ]:
# §22a — Eigenvalues (LAMA)
ev = nm.eigenvalues()
print(f"Eigenvalue subcases: {list(ev.keys())}  (one entry = all modes)")
ev[1]

In [ ]:
# §22b — Eigenvectors (OUGV1) — one subcase per mode
eig_vecs = nm.displacements()
print(f"Eigenvector modes: {list(eig_vecs.keys())}  ({len(eig_vecs[1])} DOFs each)")

# Spot-check mode 1, GRID 1 against F06
g1 = eig_vecs[1][eig_vecs[1]['GRID'] == 1].iloc[0]
print(f"\nMode 1  GRID 1  (F06 reference in brackets):")
print(f"  T1={g1.DX:+.6e}  [-1.822885E-01]")
print(f"  T2={g1.DY:+.6e}  [-6.204547E-02]")
print(f"  T3={g1.DZ:+.6e}  [-8.168588E-02]")
print(f"  R1={g1.RX:+.6e}  [-1.389719E-02]")
print(f"  R2={g1.RY:+.6e}  [+4.082382E-02]")
print(f"  R3={g1.RZ:+.6e}  [-2.415273E-06]")

In [ ]:
# §22c — Results helper summary
r_nm = nm.results(1)
r_nm

In [ ]:

# §23a — Load with geometry, inspect grids + connectivity
import importlib, op2_native.decoders.geom_dat as _gm, op2_native.reader as _rd
importlib.reload(_gm); importlib.reload(_rd)
from op2_native.reader import OP2

LIN_OP2 = 'run_files/model/cylinder-0000.op2'
lin_g = OP2(LIN_OP2, geometry=True)   # auto-finds cylinder-0000.dat

grids = lin_g.grid_coordinates()
conn  = lin_g.element_connectivity()

print(f"Grids: {len(grids):,}  (first 3)")
print(grids.head(3).to_string(index=False))
print()
for et, df in conn.items():
    print(f"  {et}: {len(df):,} elements  cols={list(df.columns)}")


In [ ]:

# §23b — Element centroids (average of corner-node coordinates)
centroids = lin_g.element_centroids('CTETRA')
print(f"CTETRA centroids: {len(centroids):,}")
print(centroids.head(5).to_string(index=False))


In [ ]:

# §23c — Join stresses to centroids: top-10 worst VM elements + their location
import pandas as pd

stresses = lin_g.solid_stresses()[1]          # subcase 1
cent_df  = lin_g.element_centroids('CTETRA')  # EID, X, Y, Z

# solid_stresses has corner output; take centroid row per element (max abs VM)
vm_col = [c for c in stresses.columns if 'VM' in c][0]
cen_vm = stresses.groupby('EID')[vm_col].max().reset_index()

top10 = (
    cen_vm.merge(cent_df, on='EID')
    .nlargest(10, vm_col)
    [['EID', vm_col, 'X', 'Y', 'Z']]
    .reset_index(drop=True)
)
print(f"Top-10 {vm_col} stress elements with centroid coordinates:")
print(top10.to_string(index=False))


In [ ]:
## §24 — Validation: plate-0001 (F06-printed vs OP2-decoded)
#
# SOL SESTATIC, 1 subcase ("LINEAR"), model: flat plate with mixed CQUAD4 + CTRIA3 elements.
# Output requests: DISPLACEMENT(PRINT)=ALL, SPCFORCE(PRINT)=ALL, OLOAD(PRINT)=ALL,
#                  FORCE(PRINT,CORNER)=ALL, STRESS(PRINT,CORNER)=ALL
#
# Reference values extracted directly from the F06 printed output.
# All tolerance checks use 0.1% relative tolerance (Float32 limited precision).


In [ ]:
# §24a — Load plate-0001.op2 and verify row counts
import importlib, op2_native.reader as _rd
importlib.reload(_rd)
from op2_native.reader import OP2

op2p = OP2(r'run_files/model/plate-0001.op2')

st   = op2p.stresses()[1]
ef   = op2p.element_forces()[1]
disp = op2p.displacements()[1]
spc  = op2p.spc_forces()[1]
olk  = op2p.applied_loads()[1]

print(f"stresses       : {len(st):>5} rows  (expect 1412 = 282 CQUAD4×5 + 2 CTRIA3)")
print(f"element_forces : {len(ef):>5} rows  (expect 1412)")
print(f"displacements  : {len(disp):>5} rows  (expect 320 grids)")
assert len(st) == 1412,   f"stress row count {len(st)} != 1412"
assert len(ef) == 1412,   f"force  row count {len(ef)} != 1412"
assert len(disp) == 320,  f"disp   row count {len(disp)} != 320"
print("Row counts OK")


In [ ]:
# §24b — Resultant equilibrium (OLOAD vs SPC forces)
import math

spc_fx  = spc['FX'].sum()
oload_fx = olk['FX'].sum()

print(f"SPC   FX sum : {spc_fx:+.4f}  (expect -1400.0)")
print(f"OLOAD FX sum : {oload_fx:+.4f}  (expect +1400.0)")

assert abs(spc_fx  + 1400.0) < 0.5, f"SPC FX {spc_fx}"
assert abs(oload_fx - 1400.0) < 0.5, f"OLOAD FX {oload_fx}"
print("Resultant equilibrium OK")


In [ ]:
# §24c — Displacements vs F06 (grids 1 and 2)
#
# F06 reference (grid 1 = fixed end, all zero; grid 2 = first free grid):
#   GRID 1: DX=0, DY=0, DZ=0, RZ=0
#   GRID 2: DX= 3.461396E-05, DY=-5.806342E-05, RZ=-2.787171E-04

g1 = disp[disp['GRID'] == 1].iloc[0]
g2 = disp[disp['GRID'] == 2].iloc[0]

TOL = 1e-6  # absolute for near-zero; relative otherwise
assert abs(g1['DX']) < TOL and abs(g1['DY']) < TOL and abs(g1['DZ']) < TOL and abs(g1['RZ']) < TOL, \
    f"GRID1 not zero: {dict(g1)}"

ref_g2 = {'DX': 3.461396e-5, 'DY': -5.806342e-5, 'RZ': -2.787171e-4}
for col, ref in ref_g2.items():
    rel_err = abs(g2[col] - ref) / abs(ref)
    assert rel_err < 1e-3, f"GRID2 {col}: got {g2[col]:.6e} vs F06 {ref:.6e} ({rel_err:.2%})"

print("GRID 1 (fixed):  DX=DY=DZ=RZ=0 ✓")
print(f"GRID 2:  DX={g2['DX']:.6e}  DY={g2['DY']:.6e}  RZ={g2['RZ']:.6e} ✓")
print("Displacement check OK")


In [ ]:
# §24d — CQUAD4 corner stresses vs F06 (EID=1)
#
# F06 reference (EID=1, corner output):
#   CEN/4 (GRID=0):  SX1=3167.859, SY1= -200.422, TXY1=3822.300, VM1=7385.143
#   GRID 27 (G272):  SX1=3803.222, SY1= -916.693, TXY1=3822.232, VM1=7913.254
#   GRID 27 (G273):  SX1=3988.173, SY1=  594.295, TXY1=3624.699, VM1=7300.951
#   GRID  7 (G72):   SX1=2415.531, SY1=  425.220, TXY1=3847.937, VM1=7029.105
#   GRID  1:         SX1=2563.312, SY1= -796.320, TXY1=3969.099, VM1=7517.124

REL = 1e-3   # 0.1% — limited by Float32 in OP2
s1 = st[st['EID'] == 1].reset_index(drop=True)

ref_stress = [
    # (GRID, SX1, SY1, TXY1, VM1)
    (0.0,  3167.859,  -200.422,  3822.300, 7385.143),
    (27.0, 3803.222,  -916.693,  3822.232, 7913.254),
    (27.0, 3988.173,   594.295,  3624.699, 7300.951),
    ( 7.0, 2415.531,   425.220,  3847.937, 7029.105),
    ( 1.0, 2563.312,  -796.320,  3969.099, 7517.124),
]

for i, (gid, sx, sy, txy, vm) in enumerate(ref_stress):
    row = s1.iloc[i]
    assert abs(row['GRID'] - gid) < 0.1, f"row {i} GRID {row['GRID']} != {gid}"
    for name, got, exp in [('SX1', row['SX1'], sx), ('SY1', row['SY1'], sy),
                            ('TXY1', row['TXY1'], txy), ('VM1', row['VM1'], vm)]:
        rel_err = abs(got - exp) / abs(exp) if exp else abs(got)
        assert rel_err < REL, f"EID1 row {i} {name}: {got:.3f} vs F06 {exp:.3f} ({rel_err:.3%})"

print("CQUAD4 EID=1 corner stress (5 rows, SX1/SY1/TXY1/VM1) vs F06 ✓")
print(s1[['EID','GRID','SX1','SY1','TXY1','VM1']].to_string(index=False))


In [ ]:
# §24e — CQUAD4 corner forces vs F06 (EID=1)
#
# F06 reference (EID=1, OPTION=BILIN corner output):
#   CEN/4 (GRID=0):  NX=411.822, NY=-26.055, NXY=496.899
#   GRID 27 (G272):  NX=494.419, NY=-119.17, NXY=496.890
#   GRID 27 (G273):  NX=518.463, NY= 77.258, NXY=471.211
#   GRID  7 (G72):   NX=314.019, NY= 55.279, NXY=500.232
#   GRID  1:         NX=333.231, NY=-103.52, NXY=515.983

REL = 1e-3
f1 = ef[ef['EID'] == 1].reset_index(drop=True)

ref_force = [
    (0.0,  411.822,  -26.055, 496.899),
    (27.0, 494.419, -119.170, 496.890),
    (27.0, 518.463,   77.258, 471.211),
    ( 7.0, 314.019,   55.279, 500.232),
    ( 1.0, 333.231, -103.522, 515.983),
]

for i, (gid, nx, ny, nxy) in enumerate(ref_force):
    row = f1.iloc[i]
    assert abs(row['GRID'] - gid) < 0.1, f"row {i} GRID {row['GRID']} != {gid}"
    for name, got, exp in [('NX', row['NX'], nx), ('NY', row['NY'], ny), ('NXY', row['NXY'], nxy)]:
        rel_err = abs(got - exp) / abs(exp) if exp else abs(got)
        assert rel_err < REL, f"EID1 row {i} {name}: {got:.3f} vs F06 {exp:.3f} ({rel_err:.3%})"

print("CQUAD4 EID=1 corner forces (5 rows, NX/NY/NXY) vs F06 ✓")
print(f1[['EID','GRID','NX','NY','NXY']].to_string(index=False))


In [ ]:
# §24f — CTRIA3 forces and stresses vs F06 (EID=63, 70)
#
# F06 reference (CTRIA3 membrane forces):
#   EID 63: FX=134.269, FY=406.877, FXY=340.964
#   EID 70: FX=132.334, FY=401.012, FXY=317.929
# (MX=MY=MXY=QX=QY=0 for membrane-only plate)

REL = 1e-3

ef_tria = ef[ef['EID'].isin([63, 70])].set_index('EID')
st_tria = st[st['EID'].isin([63, 70])].set_index('EID')

ref_tria_forces = {
    63: (134.269, 406.877, 340.964),
    70: (132.334, 401.012, 317.929),
}

for eid, (nx, ny, nxy) in ref_tria_forces.items():
    row = ef_tria.loc[eid]
    for name, got, exp in [('NX', row['NX'], nx), ('NY', row['NY'], ny), ('NXY', row['NXY'], nxy)]:
        rel_err = abs(got - exp) / abs(exp)
        assert rel_err < REL, f"CTRIA3 EID{eid} {name}: {got:.3f} vs F06 {exp:.3f} ({rel_err:.3%})"

print("CTRIA3 forces vs F06 ✓")
print(ef[ef['EID'].isin([63, 70])][['EID','NX','NY','NXY']].to_string(index=False))
print()

# CTRIA3 stresses (centroid row only, SX1/SY1/TXY1)
print("CTRIA3 stresses (centroid):")
print(st[st['EID'].isin([63, 70])][['EID','GRID','SX1','SY1','TXY1','VM1']].to_string(index=False))
print()
print("All plate-0001 validation checks passed ✓")


In [ ]:

# §25 — Nonlinear shell stresses (plate-0002, SOL 106)
#
# plate-0002 has 282 CQUAD4 + 2 CTRIA3 elements, 5 NL subcases (LOAD STEP 1–5).
# OESNLXR numwde=25 layout: 1 packed EID + 2×12 fiber words
#   fiber block: FD, SX, SY, SZ(NaN), TXY, VM, EFF_STRAIN_PLAS, EFF_CREEP,
#                EX, EY, EZ(NaN), EXY
#
# F06 reference for EID=1 (CQUAD4):
#   SUBCASE 1 (LOAD STEP=1.0)
#     FIBER 1  FD=-0.065  SX=3170.354  SY=-193.469  TXY=3824.412  VM=7387.849
#              EX=1.243923e-4  EY=-4.768024e-5  EXY=3.912667e-4
#   SUBCASE 3 (LOAD STEP=3.0, large-deformation bending)
#     FIBER 1  FD=-0.065  SX=4.452175e5  SY=1.061289e6  TXY=2.640362e5  VM=1.030143e6
#     FIBER 2  FD=+0.065  SX=-4.219273e5 SY=-1.026794e6 TXY=-2.469655e5 VM=9.909958e5

import sys; sys.path.insert(0, '.')
from op2_native import OP2

op2_nl = OP2('run_files/model/plate-0002.op2')
nl = op2_nl.nl_shell_stresses()

# basic shape checks
assert sorted(nl.keys()) == [1, 2, 3, 4, 5], f"expected 5 subcases, got {sorted(nl.keys())}"
for sc, df in nl.items():
    assert len(df) == 568, f"sc{sc}: expected 568 rows (284 EIDs x 2 fibers), got {len(df)}"
    assert df['EID'].nunique() == 284, f"sc{sc}: expected 284 unique EIDs (282 CQUAD4 + 2 CTRIA3)"

# F06 reference check for EID=1, subcase 1, FIBER 1
REL = 5e-4
ref_sc1 = {'SX': 3170.354, 'SY': -193.469, 'TXY': 3824.412, 'VM': 7387.849,
           'EX': 1.243923e-4, 'EY': -4.768024e-5, 'EXY': 3.912667e-4}

row = nl[1][(nl[1]['EID'] == 1) & (nl[1]['FIBER'] == 1)].iloc[0]
for col, exp in ref_sc1.items():
    got = row[col]
    if abs(exp) > 0:
        rel_err = abs(got - exp) / abs(exp)
        assert rel_err < REL, f"sc1 EID=1 FIBER=1 {col}: got {got:.6g} vs F06 {exp:.6g} ({rel_err:.4%})"

# F06 reference check for EID=1, subcase 3 (large-deformation bending)
ref_sc3_f1 = {'SX': 4.452175e5, 'SY': 1.061289e6, 'TXY': 2.640362e5, 'VM': 1.030143e6}
ref_sc3_f2 = {'SX': -4.219273e5, 'SY': -1.026794e6, 'TXY': -2.469655e5, 'VM': 9.909958e5}

for fiber, ref in [(1, ref_sc3_f1), (2, ref_sc3_f2)]:
    row3 = nl[3][(nl[3]['EID'] == 1) & (nl[3]['FIBER'] == fiber)].iloc[0]
    for col, exp in ref.items():
        got = row3[col]
        rel_err = abs(got - exp) / abs(exp)
        assert rel_err < REL, f"sc3 EID=1 FIBER={fiber} {col}: got {got:.6g} vs F06 {exp:.6g} ({rel_err:.4%})"

# CTRIA3 elements present
assert {63, 70}.issubset(set(nl[1]['EID'].unique())), "CTRIA3 EIDs 63/70 missing from sc1"

# sc3 fibers have opposite sign (bending-dominated response)
f1 = nl[3][(nl[3]['EID'] == 1) & (nl[3]['FIBER'] == 1)].iloc[0]
f2 = nl[3][(nl[3]['EID'] == 1) & (nl[3]['FIBER'] == 2)].iloc[0]
assert f1['SX'] * f2['SX'] < 0, "sc3 EID=1 SX fibers expected to have opposite sign (bending)"

print("nl_shell_stresses: all checks passed")
print(f"  subcases: {sorted(nl.keys())}  rows/subcase: {len(nl[1])}")
print(f"  sc1 EID=1 FIBER=1  SX={nl[1][(nl[1].EID==1) & (nl[1].FIBER==1)].iloc[0]['SX']:.3f}  VM={nl[1][(nl[1].EID==1) & (nl[1].FIBER==1)].iloc[0]['VM']:.3f}")
print(f"  sc3 EID=1 FIBER=1  SX={nl[3][(nl[3].EID==1) & (nl[3].FIBER==1)].iloc[0]['SX']:.3f}  VM={nl[3][(nl[3].EID==1) & (nl[3].FIBER==1)].iloc[0]['VM']:.3f}")
print(f"  sc3 EID=1 FIBER=2  SX={nl[3][(nl[3].EID==1) & (nl[3].FIBER==2)].iloc[0]['SX']:.3f}  VM={nl[3][(nl[3].EID==1) & (nl[3].FIBER==2)].iloc[0]['VM']:.3f}")


In [5]:
import pandas as pd
from op2_native import OP2

op2_nl = OP2('run_files/model/plate-0002.op2')
nl = op2_nl.stress_tensors(location='max')

pd.DataFrame(nl[1])
# nl


,EID,SX1,SY1,TXY1,SX2,SY2,TXY2
0,1,3990.857178,599.581665,3971.085449,3990.857178,599.581665,3971.085449
1,2,5646.335449,480.848724,566.461182,5646.335449,480.848724,566.461182
2,3,1881.109741,2296.273193,3857.526611,1881.109741,2296.273193,3857.526611
3,4,4789.971191,-127.394516,2573.572754,4789.971191,-127.394516,2573.572754
4,5,5483.299805,-690.195679,697.243042,5483.299805,-690.195679,697.243042
...,...,...,...,...,...,...,...
279,282,2608.709473,1937.046753,-3159.387695,2608.709473,1937.046753,-3159.387695
280,283,1473.224365,3069.679199,-1832.102295,1473.224365,3069.679199,-1832.102295
281,284,-6.588562,5076.298340,-439.901733,-6.588562,5076.298340,-439.901733
282,63,1032.227783,3127.962891,2622.213867,1032.227783,3127.962891,2622.213867


In [1]:
pd.DataFrame(op2_nl.stress_tensors()[1])

NameError: name 'pd' is not defined